# Notebook 02 — Admin Services

**Prerequisites**
- Notebook 01 completed (DB seeded with users + roles)

**What this validates**
- Register a document (metadata only)
- Assign role access to document
- `get_accessible_docs()` with union logic
- Archive toggle
- No-access case
- Multi-role union access

In [1]:
import sys
sys.path.insert(0, '..')

import uuid
from sqlalchemy import select

from backend.db.session import get_session
from backend.db.models import User, Role, UserRole, Document, DocumentAccess
from backend.admin import user_service, document_service, access_service

print('Imports OK')

Imports OK


## 1. Register a Document (metadata only)

In [2]:
async def test_register_document():
    async with get_session() as session:
        # Get admin user
        admin = (await session.execute(select(User).where(User.email == 'admin@example.com'))).scalar_one()

        # Register IT policy doc (idempotent)
        existing_it = (await session.execute(
            select(Document).where(Document.title == 'IT Security Policy v2.1')
        )).scalar_one_or_none()
        if existing_it is None:
            it_doc = await document_service.register_document(
                session,
                title='IT Security Policy v2.1',
                storage_uri='file:///data/uploads/it_security_policy.pdf',
                uploaded_by=admin.user_id,
            )
        else:
            it_doc = existing_it
        print(f'IT doc: {it_doc.title} (id={it_doc.doc_id})')

        # Register Finance policy doc (idempotent)
        existing_fin = (await session.execute(
            select(Document).where(Document.title == 'Finance Compliance Policy 2024')
        )).scalar_one_or_none()
        if existing_fin is None:
            fin_doc = await document_service.register_document(
                session,
                title='Finance Compliance Policy 2024',
                storage_uri='file:///data/uploads/finance_compliance.pdf',
                uploaded_by=admin.user_id,
            )
        else:
            fin_doc = existing_fin
        print(f'Finance doc: {fin_doc.title} (id={fin_doc.doc_id})')

        return it_doc.doc_id, fin_doc.doc_id

it_doc_id, fin_doc_id = await test_register_document()
print('Document registration: PASS')

IT doc: IT Security Policy v2.1 (id=db3e8177-e4f3-4e81-9f99-33099a666f08)
Finance doc: Finance Compliance Policy 2024 (id=2d223339-54c2-48f9-846e-75d2960eb419)
Document registration: PASS


## 2. Assign Role Access

In [3]:
async def test_assign_access():
    async with get_session() as session:
        it_role = (await session.execute(select(Role).where(Role.role_name == 'it_eng'))).scalar_one()
        fin_role = (await session.execute(select(Role).where(Role.role_name == 'finance'))).scalar_one()
        it_audit_role = (await session.execute(select(Role).where(Role.role_name == 'it_auditor'))).scalar_one()
        
        # IT doc accessible to IT engineers and IT auditor
        await access_service.set_document_access(session, it_doc_id, [it_role.role_id, it_audit_role.role_id])
        print(f'IT doc assigned to: it_eng, it_auditor')
        
        # Finance doc accessible to finance role only
        await access_service.set_document_access(session, fin_doc_id, [fin_role.role_id])
        print(f'Finance doc assigned to: finance')

await test_assign_access()
print('Role access assignment: PASS')

IT doc assigned to: it_eng, it_auditor
Finance doc assigned to: finance
Role access assignment: PASS


## 3. get_accessible_docs() — Union Logic

In [4]:
async def test_accessible_docs():
    async with get_session() as session:
        it_user = (await session.execute(select(User).where(User.email == 'it.user@example.com'))).scalar_one()
        fin_user = (await session.execute(select(User).where(User.email == 'finance.user@example.com'))).scalar_one()
        auditor = (await session.execute(select(User).where(User.email == 'auditor@example.com'))).scalar_one()
        multi_user = (await session.execute(select(User).where(User.email == 'multi@example.com'))).scalar_one()

        it_docs = await access_service.get_accessible_docs(session, it_user.user_id)
        fin_docs = await access_service.get_accessible_docs(session, fin_user.user_id)
        auditor_docs = await access_service.get_accessible_docs(session, auditor.user_id)
        multi_docs = await access_service.get_accessible_docs(session, multi_user.user_id)

        print(f'IT user sees:              {len(it_docs)} docs')
        print(f'Finance user sees:         {len(fin_docs)} docs')
        print(f'Global Auditor sees:       {len(auditor_docs)} docs (all)')
        print(f'Multi-role user sees:      {len(multi_docs)} docs')

        # IT user (it_eng role) sees the notebook test doc
        it_titles = [d.title for d in it_docs]
        assert any('IT' in t for t in it_titles), f'IT user should see at least one IT doc, got: {it_titles}'

        # Finance user (finance role) sees the notebook test doc
        fin_titles = [d.title for d in fin_docs]
        assert any('Finance' in t for t in fin_titles), f'Finance user should see at least one Finance doc, got: {fin_titles}'

        # Global Auditor sees everything
        assert len(auditor_docs) >= 2, f'Global auditor should see all docs, got {len(auditor_docs)}'

        # Multi-role user (IT + Finance) sees union of both
        assert len(multi_docs) >= 2, f'Multi-role user should see at least 2 docs, got {len(multi_docs)}'

        print('Assertions passed.')

await test_accessible_docs()
print('get_accessible_docs union logic: PASS')

IT user sees:              1 docs
Finance user sees:         1 docs
Global Auditor sees:       49 docs (all)
Multi-role user sees:      2 docs
Assertions passed.
get_accessible_docs union logic: PASS


## 4. Archive Toggle

In [5]:
async def test_archive_toggle():
    async with get_session() as session:
        it_user = (await session.execute(select(User).where(User.email == 'it.user@example.com'))).scalar_one()

        # Count baseline docs before archive
        before = await access_service.get_accessible_docs(session, it_user.user_id)
        baseline = len(before)

        # Archive IT test doc
        await document_service.archive_document(session, it_doc_id)
        await session.flush()  # autoflush=False requires explicit flush

        # Without include_archived: should drop by 1
        docs_no_archive = await access_service.get_accessible_docs(session, it_user.user_id, include_archived=False)
        print(f'After archive, without toggle: {len(docs_no_archive)} docs (was {baseline})')
        assert len(docs_no_archive) == baseline - 1, f'Expected {baseline - 1} docs, got {len(docs_no_archive)}'

        # With include_archived: back to baseline
        docs_with_archive = await access_service.get_accessible_docs(session, it_user.user_id, include_archived=True)
        print(f'After archive, with toggle:    {len(docs_with_archive)} docs')
        assert len(docs_with_archive) == baseline

        # Unarchive — restore to baseline
        await document_service.unarchive_document(session, it_doc_id)
        await session.flush()
        docs_restored = await access_service.get_accessible_docs(session, it_user.user_id)
        assert len(docs_restored) == baseline
        print('Archive restored')

await test_archive_toggle()
print('Archive toggle: PASS')

After archive, without toggle: 0 docs (was 1)
After archive, with toggle:    1 docs
Archive restored
Archive toggle: PASS


## 5. No-Access Case

In [6]:
async def test_no_access():
    async with get_session() as session:
        # Finance user should not see IT doc
        fin_user = (await session.execute(select(User).where(User.email == 'finance.user@example.com'))).scalar_one()
        docs = await access_service.get_accessible_docs(session, fin_user.user_id)
        titles = [d.title for d in docs]
        print(f'Finance user sees: {titles}')
        assert not any('IT' in t for t in titles), 'Finance user should not see IT docs'
        
        # Create user with no roles
        from backend.db.models import ResponseFormat
        from backend.auth.password import hash_password
        existing = (await session.execute(select(User).where(User.email == 'norole@example.com'))).scalar_one_or_none()
        if existing is None:
            norole_user = User(email='norole@example.com', password_hash=hash_password('pass'), default_format=ResponseFormat.EXECUTIVE_SUMMARY)
            session.add(norole_user)
            await session.flush()
        else:
            norole_user = existing
        
        no_docs = await access_service.get_accessible_docs(session, norole_user.user_id)
        print(f'No-role user sees: {[d.title for d in no_docs]}')
        assert len(no_docs) == 0

await test_no_access()
print('No-access case: PASS')

Finance user sees: ['Finance Compliance Policy 2024']
No-role user sees: []
No-access case: PASS


## Summary

All Phase 2 (Admin) checks passed:
- Document registration (metadata only)
- Role access assignment
- Union-of-roles access query
- Archive toggle (include/exclude)
- No-access case returns empty list
- Multi-role union works correctly

**Next:** Notebook 03 — Document Parsing (user-owned RAG exploration)